In [0]:
%sql
-- Databricks notebook source
-- Crea la silver desde bronze: fecha a DATE, horas a TIMESTAMP y órdenes que cruzan la medianoche.
-- Es el primer paso del 9 de octubre, tal cual. Después van st_ventas_unidades y 03_silver_limpieza.

CREATE OR REPLACE TABLE workspace.default.silver_ventas AS
WITH t AS (
  SELECT b.* EXCEPT (date, begin_time, end_time),
         CAST(try_to_timestamp(b.date, 'M/d/yyyy') AS DATE) AS date,
         try_to_timestamp(concat(b.date, ' ', b.begin_time), 'M/d/yyyy h:mm a') AS begin_time,
         try_to_timestamp(concat(b.date, ' ', b.end_time), 'M/d/yyyy h:mm a') AS end_time
  FROM workspace.default.bronze_ventas b
)
SELECT * EXCEPT (end_time),
       CASE WHEN timestampdiff(MINUTE, begin_time, end_time) < -720
            THEN end_time + INTERVAL 1 DAY
            ELSE end_time END AS end_time
FROM t